# 03C — Automatic spot tracking  ·  line B (DS0N)

`03B` separates the spots of a region by hand — one copied block per spot, a `CLUSTER_XLIM`
typed in pixels. Here the region goes in raw and `src/tracking.py` does the rest: it finds up to
four spots in the first frame, follows each by pixel overlap, names the umbra splits
(`DS00_A` → `DS00_AA`), shares the penumbra until it splits too, resolves merges, filters, and
then runs **the same steps 2–6 as `03B`** on every spot that survives.

The rules are in the docstring of `src/tracking.py`; every threshold is a field of
`tracking.TrackingParams`, set in the cell below.

| name | meaning |
|---|---|
| `DS00_A`…`_D` | the (up to 4) spots detected in the first frame, largest first |
| `DS00_AA` | split off `DS00_A`: umbra first, then its own penumbra |
| `DS00_AA'` | split off `DS00_A` in the umbra only — it shares `A`'s penumbra (folder `DS00_AAp`) |

Output goes to `data/processed/sebastian_sun_spots/testing/`, never over the `03B` results:
one `<DS>/` folder per region (`spots.csv`, `events.csv`, `masks.npz`, the tracking overview),
one `<DS>_<spot>/` folder per spot laid out like a `03B` one, and `anim_sumary_arch.html`.

**Look at each region's tracking overview before believing any per-spot number.**

In [ ]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import pandas as pd
from IPython.display import FileLink, Image, display

from src import config, report, tracking

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

## Knobs for this run

In [ ]:
PARAMS = tracking.TrackingParams(
    max_initial_spots=4,     # spots taken in the first frame
    min_spot_px=100,         # footprint (umbra + penumbra) to count as a spot; ~4 MSH
    min_umbra_split_px=80,   # a separated umbra fragment must be at least this big ...
    min_split_fraction=0.25, # ... and this fraction of its parent's umbra ...
    min_split_frames=20,     # ... and stay apart this many frames, ~4 h (also: merges, penumbra splits)
    min_duration_h=12.0,     # penumbra-split children shorter than this are dropped
    heal_within_h=24.0,      # umbra-only splits that rejoin within this are annulled
)
N_DATASETS   = 10            # the first N raw datasets that load
RUN_PIPELINE = True          # False: track and write the overview only (fast, for tuning)
OUT_ROOT     = config.DS0N_PROCESSED_DIR / 'testing'

print(PARAMS)
print(f'output -> {OUT_ROOT}')

## Track every region, then run steps 2–6 on every kept spot

A dataset that does not load (a truncated or missing cube) is reported and skipped, and the
next one is taken instead — the run stops after `N_DATASETS` regions that did load.

In [ ]:
results, skipped = {}, {}
for ds_dir in config.ds0n_regions():
    if len(results) >= N_DATASETS:
        break
    print(f'\n=================== {ds_dir.name} ===================')
    try:
        results[ds_dir.name] = tracking.process_region(ds_dir, OUT_ROOT, PARAMS,
                                                       run_pipeline=RUN_PIPELINE)
    except Exception as exc:          # noqa: BLE001 — a broken delivery is skipped, not fatal
        skipped[ds_dir.name] = f'{type(exc).__name__}: {exc}'
        print(f'SKIPPED {ds_dir.name}: {skipped[ds_dir.name]}')

print(f'\nprocessed: {list(results)}')
print(f'skipped:   {skipped}')

## Tracking overview per region

In [ ]:
for name, result in results.items():
    print(f'\n{name}: {result.table.kept.sum()} kept / {len(result.table)} tracked')
    display(result.table[['spot', 'parent', 'time_start_h', 'time_end_h', 'duration_h',
                          'median_area_px', 'median_umbra_px', 'penumbra_split',
                          'end_reason', 'kept', 'reject_reason', 'pipeline_error']])
    display(Image(filename=str(OUT_ROOT / name / 'plots' / 'tracking_overview.png')))

## All spots, one table, and the summary page

In [ ]:
all_spots = pd.concat([r.table.assign(dataset=n) for n, r in results.items()], ignore_index=True)
all_spots.to_csv(OUT_ROOT / 'spots_all.csv', index=False)
print(f"{int(all_spots.kept.sum())} kept spots of {len(all_spots)} tracked in {len(results)} regions")
print(all_spots.groupby('dataset').agg(tracked=('spot', 'size'), kept=('kept', 'sum')))

summary = report.build_ds0n_summary(processed_dir=OUT_ROOT,
                                    title='Sun Spot Pulsation Review — DS0N automatic spot tracking (testing)')
FileLink(str(summary.relative_to(project_root)))